# Path 2: Python Code Snippet - Stored Procedure + Task

In [ ]:
%%sql -r Stored_Procedure
CREATE OR REPLACE PROCEDURE ML_LAB.DATA.sproc_inference()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-ml-python', 'snowflake-snowpark-python')
HANDLER = 'run_inference'
AS
$$
from snowflake.ml.registry import Registry

def run_inference(session):
    reg = Registry(session=session, database_name="ML_LAB", schema_name="DATA")
    model = reg.get_model("iris_classifier").version("v1")

    input_df = session.table("ML_LAB.DATA.IRIS").drop("SPECIES")
    predictions = model.run(input_df, function_name="predict")

    predictions.write.save_as_table(
        "ML_LAB.DATA.IRIS_PREDICTIONS",
        mode="overwrite"
    )
    return "Inference complete. Results written to ML_LAB.DATA.IRIS_PREDICTIONS."
$$;

In [ ]:
CALL run_iris_inference();
-- Returns: "Inference complete. Results written to ML_LAB.DATA.IRIS_PREDICTIONS."

In [ ]:
CREATE OR REPLACE TASK ML_LAB.DATA.sproc_inference_task
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 7 * * * UTC'
AS
  CALL ML_LAB.DATA.sproc_inference();

ALTER TASK ML_LAB.DATA.sproc_inference_task RESUME;